<a href="https://colab.research.google.com/github/Jeyoon-hwang/portfolio/blob/master/AI_%EC%9E%85%EB%B2%95_%EC%88%98%EC%9A%94_%EB%B6%84%EC%84%9D%EC%9D%84_%EC%9C%84%ED%95%9C_%EC%A0%84%EC%B2%98%EB%A6%AC_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

# ==============================================================================
# 1. 하이브리드 사용자 사전 정의 (가상)
# 실제 환경에서는 대규모 사전 파일을 로드하여 사용합니다.
# ==============================================================================

# 1-1. 법률 용어 사전 (정확성 및 보존 목적)
# 형태소 분석기가 '민법'을 '민'+'법'으로 분리하지 않도록 고유명사로 등록합니다.
legal_dictionary = {
    "민법": "NNP",
    "형소법": "NNP",
    "공정거래위원회": "NNP",
    "대한민국헌법": "NNP"
}

# 1-2. 신조어/은어 사전 (정규화 및 표준화 목적)
# 비표준어를 표준적인 의미로 변환하거나, 분석을 위해 태깅합니다.
slang_dictionary = {
    "JMT": "아주 맛있다",
    "ㅇㅈ": "인정",
    "갑분싸": "갑자기 분위기가 싸해진다",
    "강추템": "강력 추천 아이템"
}

# 1-3. 하이브리드 사전 결합
# Mecab 같은 형태소 분석기에 적용할 통합 사전을 만듭니다.
hybrid_dictionary = {**legal_dictionary, **slang_dictionary}


# ==============================================================================
# 2. 이원적 전처리 파이프라인 (Dual-Path Preprocessing Pipeline)
# ==============================================================================

class PreprocessingPipeline:
    def __init__(self, custom_dictionary):
        """
        파이프라인 초기화. 사용자 정의 사전을 받습니다.
        """
        self.dictionary = custom_dictionary
        # 이모티콘 및 특수문자 제거를 위한 정규식 패턴
        self.special_char_pattern = re.compile(r'[^가-힣A-Za-z0-9\s_제항조\.]')
        # 반복되는 자음/모음 제거를 위한 패턴 (예: ㅋㅋㅋㅋ -> ㅋ)
        self.repeated_char_pattern = re.compile(r'([ㄱ-ㅎㅏ-ㅣ])\1{2,}')

    # --------------------------------------------------------------------------
    # 경로 1: 법률 텍스트 파이프라인 (Legal Text Pipeline)
    # --------------------------------------------------------------------------
    def process_legal_text(self, text: str) -> str:
        """
        법률 텍스트에 특화된 전처리 함수. 구조적 정보 보존에 집중합니다.
        Args:
            text (str): 원본 법률 뉴스 텍스트
        Returns:
            str: 전처리된 텍스트
        """
        print("\n--- [경로 1] 법률 텍스트 처리 시작 ---")

        # 1단계: 법 조항 패턴 보존 (예: '제1조 제2항' -> '제_1조_제_2항_')
        # 형태소 분석 시 분리되지 않도록 특수한 형태로 변환합니다.
        processed_text = re.sub(r'제(\s*)(\d+)(\s*)조', r'제_\2조_', text)
        processed_text = re.sub(r'제(\s*)(\d+)(\s*)항', r'제_\2항_', processed_text)
        print(f"1. 법 조항 처리: {processed_text}")

        # 2단계: 불필요한 공백 제거
        processed_text = ' '.join(processed_text.split())
        print(f"2. 공백 정규화: {processed_text}")

        # 3단계: (시뮬레이션) 하이브리드 사전을 이용한 형태소 분석
        # 실제로는 Mecab.morphs(text, user_dic=...) 등을 사용합니다.
        # 여기서는 사전의 단어가 포함되어 있는지 확인하는 것으로 대체합니다.
        tokens = self.tokenize_simulation(processed_text)
        print(f"3. 형태소 분석 (시뮬레이션): {tokens}")

        print("--- 법률 텍스트 처리 완료 ---")
        return ' '.join(tokens)

    # --------------------------------------------------------------------------
    # 경로 2: 소셜 데이터 파이프라인 (Social Data Pipeline)
    # --------------------------------------------------------------------------
    def process_social_text(self, text: str) -> str:
        """
        소셜 미디어 텍스트에 특화된 전처리 함수. 노이즈 제거에 집중합니다.
        Args:
            text (str): 원본 소셜 미디어 텍스트
        Returns:
            str: 전처리된 텍스트
        """
        print("\n--- [경로 2] 소셜 데이터 처리 시작 ---")

        # 1단계: 신조어/은어 변환
        for slang, standard in self.dictionary.items():
            # JMT, ㅇㅈ 등은 문자 그대로 치환
            if slang in text:
                 text = text.replace(slang, standard)
        processed_text = text
        print(f"1. 신조어 처리: {processed_text}")

        # 2단계: 맞춤법 교정 (시뮬레이션)
        # 실제 환경에서는 'hanspell' 같은 라이브러리 사용을 권장합니다.
        # 예: from hanspell import spell_checker; spell_checker.check(text).checked
        processed_text = self.spell_check_simulation(processed_text)
        print(f"2. 맞춤법 교정 (시뮬레이션): {processed_text}")

        # 3단계: 반복 문자 정규화 (예: ㅋㅋㅋㅋㅋ -> ㅋㅋ)
        processed_text = self.repeated_char_pattern.sub(r'\1\1', processed_text)
        print(f"3. 반복 문자 처리: {processed_text}")

        # 4단계: 이모티콘 및 특수문자 제거
        processed_text = self.special_char_pattern.sub(' ', processed_text)
        print(f"4. 특수 문자/이모티콘 제거: {processed_text}")

        # 5단계: 불필요한 공백 제거
        processed_text = ' '.join(processed_text.split())
        print(f"5. 공백 정규화: {processed_text}")

        # 6단계: (시뮬레이션) 하이브리드 사전을 이용한 형태소 분석
        tokens = self.tokenize_simulation(processed_text)
        print(f"6. 형태소 분석 (시뮬레이션): {tokens}")

        print("--- 소셜 데이터 처리 완료 ---")
        return ' '.join(tokens)

    # --------------------------------------------------------------------------
    # 시뮬레이션 및 보조 함수들
    # --------------------------------------------------------------------------
    def spell_check_simulation(self, text: str) -> str:
        """맞춤법 교정기 동작을 흉내 내는 함수"""
        # 간단한 오타 교정 규칙
        corrections = {"마싯다": "맛있다", "개정안이 통과됬다": "개정안이 통과됐다"}
        for wrong, right in corrections.items():
            text = text.replace(wrong, right)
        return text

    def tokenize_simulation(self, text: str) -> list:
        """
        Mecab과 사용자 사전을 이용한 형태소 분석을 흉내 내는 함수.
        사전에 있는 단어는 분리하지 않고, 나머지는 공백 기준으로 분리합니다.
        """
        # 사용자 사전에 있는 단어를 임시 태그로 치환
        # 예: '공정거래위원회' -> '__DIC_WORD_0__'
        tagged_text = text
        dic_words = []
        for word in self.dictionary.keys():
            # Use re.escape to handle special characters in dictionary words
            pattern = r'\b' + re.escape(word) + r'\b'
            if re.search(pattern, tagged_text):
                tag = f"__DIC_WORD_{len(dic_words)}__"
                # Replace only the first occurrence to avoid issues with overlapping words
                tagged_text = re.sub(pattern, tag, tagged_text, 1)
                dic_words.append(word)


        # 공백으로 기본 토큰화
        tokens = tagged_text.split()

        # 임시 태그를 원래 단어로 복원
        final_tokens = []
        for token in tokens:
            if token.startswith("__DIC_WORD_"):
                parts = token.split('_')
                if len(parts) >= 3:  # Ensure there are enough parts after splitting
                    try:
                        index = int(parts[-2])
                        if 0 <= index < len(dic_words): # Ensure index is within bounds
                            final_tokens.append(dic_words[index])
                        else:
                             final_tokens.append(token) # Append original token if index is out of bounds
                    except ValueError:
                        final_tokens.append(token) # Append original token if conversion to int fails
                else:
                    final_tokens.append(token) # Append original token if not enough parts
            else:
                final_tokens.append(token)
        return final_tokens


# ==============================================================================
# 3. 파이프라인 실행 예제 (Demonstration)
# ==============================================================================
if __name__ == "__main__":
    # 샘플 데이터 정의
    legal_news_sample = "공정거래위원회는 민법 제 110조 제 1항에 의거하여 새로운 규제안을 발표했다."
    social_media_sample = "이번 개정안이 통과됬다니...😂 이건 진짜 ㅇㅈ이지ㅋㅋㅋ 마싯다 JMT!!👍"

    print("="*50)
    print("AI 입법 수요 분석을 위한 이원적 전처리 파이프라인")
    print("="*50)

    # 파이프라인 인스턴스 생성 (하이브리드 사전 전달)
    pipeline = PreprocessingPipeline(hybrid_dictionary)

    # 각 파이프라인 실행
    final_legal_text = pipeline.process_legal_text(legal_news_sample)
    final_social_text = pipeline.process_social_text(social_media_sample)

    print("\n\n" + "="*50)
    print("최종 결과")
    print("="*50)
    print(f"법률 뉴스 원본: {legal_news_sample}")
    print(f"처리 후 결과: {final_legal_text}\n")
    print(f"소셜 미디어 원본: {social_media_sample}")
    print(f"처리 후 결과: {final_social_text}")

AI 입법 수요 분석을 위한 이원적 전처리 파이프라인

--- [경로 1] 법률 텍스트 처리 시작 ---
1. 법 조항 처리: 공정거래위원회는 민법 제_110조_ 제_1항_에 의거하여 새로운 규제안을 발표했다.
2. 공백 정규화: 공정거래위원회는 민법 제_110조_ 제_1항_에 의거하여 새로운 규제안을 발표했다.
3. 형태소 분석 (시뮬레이션): ['공정거래위원회는', '__DIC_WORD_0__', '제_110조_', '제_1항_에', '의거하여', '새로운', '규제안을', '발표했다.']
--- 법률 텍스트 처리 완료 ---

--- [경로 2] 소셜 데이터 처리 시작 ---
1. 신조어 처리: 이번 개정안이 통과됬다니...😂 이건 진짜 인정이지ㅋㅋㅋ 마싯다 아주 맛있다!!👍
2. 맞춤법 교정 (시뮬레이션): 이번 개정안이 통과됐다니...😂 이건 진짜 인정이지ㅋㅋㅋ 맛있다 아주 맛있다!!👍
3. 반복 문자 처리: 이번 개정안이 통과됐다니...😂 이건 진짜 인정이지ㅋㅋ 맛있다 아주 맛있다!!👍
4. 특수 문자/이모티콘 제거: 이번 개정안이 통과됐다니...  이건 진짜 인정이지   맛있다 아주 맛있다   
5. 공백 정규화: 이번 개정안이 통과됐다니... 이건 진짜 인정이지 맛있다 아주 맛있다
6. 형태소 분석 (시뮬레이션): ['이번', '개정안이', '통과됐다니...', '이건', '진짜', '인정이지', '맛있다', '아주', '맛있다']
--- 소셜 데이터 처리 완료 ---


최종 결과
법률 뉴스 원본: 공정거래위원회는 민법 제 110조 제 1항에 의거하여 새로운 규제안을 발표했다.
처리 후 결과: 공정거래위원회는 __DIC_WORD_0__ 제_110조_ 제_1항_에 의거하여 새로운 규제안을 발표했다.

소셜 미디어 원본: 이번 개정안이 통과됬다니...😂 이건 진짜 ㅇㅈ이지ㅋㅋㅋ 마싯다 JMT!!👍
처리 후 결과: 이번 개정안이 통과됐다니... 이건 진짜 인정이지 맛있다 아주 맛있다
